In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('..'))

import torch
import math
from transformers import AutoTokenizer
from transformers.configuration_utils import PretrainedConfig
from transformers.utils import logging
from transformers import AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset

# Baseline DeepSeekMoE
from source.deepseek_baseline.config import DeepseekConfig as BaselineConfig
from source.deepseek_baseline.model import DeepseekForCausalLM as BaselineModel

# DYNMoE baseline (pure DYNMoE architecture)
from source.dynmoe_baseline.config import DynMoEConfig as DYNMoEBaseConfig
from source.dynmoe_baseline.model import DynMoEForCausalLM as DYNMoEBaseModel
from source.dynmoe_baseline.adaptive_tuning import AdaptiveExpertTuningCallback

# Prototype: DeepSeekMoE with DYNMoE routing
from source.deepseek_dynamics_routing.config import DeepseekConfig as DynmoeConfig
from source.deepseek_dynamics_routing.model import DeepseekForCausalLM as DynmoeModel
from source.deepseek_dynamics_routing.adaptive_tuning import AdaptiveExpertTuningCallback

# Utilities
from source.training_utils.monitoring import ResourceMonitorCallback, MoEMetricsCallback
from source.training_utils.save_model import save_model_and_tokenizer
from source.training_utils.summarization import print_training_summary
from source.data_preprocessing import load_and_preprocess_multiwoz

torch.cuda.empty_cache()
world_size = torch.cuda.device_count()
print(f"Number of GPUs: {world_size}")

In [ ]:
# Shared training hyperparameters
OUTPUT_DIR_BASELINE = "./checkpoints/baseline"
OUTPUT_DIR_DYNMOE_BASE = "./checkpoints/dynmoe_baseline"
OUTPUT_DIR_DYNMOE_ROUTING = "./checkpoints/dynmoe_routing"

In [ ]:
# Load data
train_sequences, val_sequences, test_sequences = load_and_preprocess_multiwoz(
    zip_path="MultiWOZ-coref/MultiWOZ2_3.zip",
    sample_size=300,
    random_seed=42
)

print(f"Train sequences: {len(train_sequences)}")
print(f"Validation sequences: {len(val_sequences)}")
print(f"Test sequences: {len(test_sequences)}")

In [ ]:
MAX_SEQ_LEN = 256
PER_DEVICE_BATCH = 8
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4
NUM_EPOCHS = 1
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# DeepSpeed config
ds_config = {
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "fp16": {"enabled": True, "loss_scale": 0, "initial_scale_power": 16, "hysteresis": 2, "min_loss_scale": 1},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "offload_param": {"device": "cpu", "pin_memory": True},
        "overlap_comm": True,
        "contiguous_gradients": True,
        "reduce_bucket_size": "auto",
        "stage3_prefetch_bucket_size": "auto",
        "stage3_param_persistence_threshold": "auto",
        "stage3_max_live_parameters": 1e9,
        "stage3_max_reuse_distance": 1e9,
        "stage3_gather_16bit_weights_on_model_save": True
    },
    "gradient_clipping": 1.0,
    "steps_per_print": 10,
    "wall_clock_breakdown": False
}

print("Configuration loaded")

In [ ]:
def train_model(ModelClass, ConfigClass, output_dir, is_dynmoe=False):
    tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-moe-16b-base", use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    config = ConfigClass()
    model = ModelClass(config)
    model.resize_token_embeddings(len(tokenizer))
    model = model.to("cuda:0")
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model.train()
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")
    
    dataset = load_dataset("text", data_files={"train": "train_sequences.txt", "validation": "val_sequences.txt"})
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LEN, padding=False, return_attention_mask=True)
    
    tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"], num_proc=2)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        fp16=True,
        logging_steps=10,
        evaluation_strategy="epoch",
        report_to="none",
        deepspeed=ds_config,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        dataloader_num_workers=2,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        seed=42
    )
    
    resource_monitor = ResourceMonitorCallback(PER_DEVICE_BATCH, world_size, GRAD_ACCUM, MAX_SEQ_LEN)
    moemetrics = MoEMetricsCallback(
        tokenized_datasets["validation"], tokenizer, data_collator,
        early_stop_patience=EARLY_STOPPING_PATIENCE,
        early_stop_threshold=EARLY_STOPPING_THRESHOLD
    )
    
    callbacks = [resource_monitor, moemetrics]
    if is_dynmoe:
        adaptive_callback = AdaptiveExpertTuningCallback(audit_steps=10)
        callbacks.append(adaptive_callback)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        data_collator=data_collator,
        tokenizer=tokenizer,
        callbacks=callbacks
    )
    
    print("Starting training...")
    train_result = trainer.train()
    print("Training finished.")
    
    eval_results = trainer.evaluate()
    final_loss = eval_results.get("eval_loss", float('inf'))
    perplexity = math.exp(final_loss) if 0 < final_loss < 30 else float('inf')
    print(f"Final validation loss: {final_loss:.4f}")
    print(f"Validation Perplexity: {perplexity:.2f}")
    
    save_model_and_tokenizer(trainer, output_dir)
    print_training_summary(resource_monitor, moemetrics, train_result, eval_results, perplexity)
    
    return trainer

print("Training function defined")

In [ ]:
# Run Baseline (DeepSeekMoE)
print("=" * 60)
print("TRAINING DeepSeekMoE (BASELINE)")
print("=" * 60)
baseline_trainer = train_model(BaselineModel, BaselineConfig, OUTPUT_DIR_BASELINE, is_dynmoe=False)

In [ ]:
# Run DYNMoE Baseline (pure DYNMoE)
print("=" * 60)
print("TRAINING DYNMoE (BASELINE)")
print("=" * 60)
dynmoe_base_trainer = train_model(DYNMoEBaseModel, DYNMoEBaseConfig, OUTPUT_DIR_DYNMOE_BASE, is_dynmoe=True)

In [ ]:
# Run Prototype: DeepSeekMoE with DYNMoE routing
print("=" * 60)
print("TRAINING DeepSeekMoE DYNMoE Routing (PROTOTYPE)")
print("=" * 60)
dynmoe_routing_trainer = train_model(DynmoeModel, DynmoeConfig, OUTPUT_DIR_DYNMOE_ROUTING, is_dynmoe=True)